# Ngày 11 — Agent & Tool/Function CallingCho LLM tự quyết định **gọi công cụ** (truy vấn DB, tra chính sách, tính toán) thay vì bịa.> Khung sẵn — điền phần `# TODO`. Yêu cầu: agent gọi đúng tool cho ≥5 câu (≥1 câu cần ≥2 tool, ≥1 câu không cần tool).

## Chuẩn bị

In [ ]:
import osimport reimport jsonimport timeimport sqlite3import pandas as pdfrom openai import OpenAIfrom dotenv import load_dotenvload_dotenv()client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))MODEL = "gpt-4o-mini"

## Nạp dữ liệu dùng chung- Bảng `users` (SQLite) từ Ngày 4 — mentor đặt file CSV nội bộ vào `docs/`.- RAG "Sổ tay chính sách IT" từ Ngày 9 (`retrieve`).

In [ ]:
# users -> SQLite (dùng lại dataset Ngày 4). Không commit file CSV nội bộ.CSV_PATH = "users_7_7_2026 2_49_38 AM.csv"  # đặt cạnh notebook trong docs/conn = sqlite3.connect(":memory:")df_users = pd.read_csv(CSV_PATH)df_users.to_sql("users", conn, index=False, if_exists="replace")# RAG Ngày 9: dùng lại hàm retrieve()%run day9.ipynb

## 1) Định nghĩa các tool (hàm Python thật)

In [ ]:
def query_users_db(sql: str):    """Chạy 1 truy vấn CHỈ-ĐỌC trên bảng users, trả về list dict."""    # TODO: chặn mọi câu không phải SELECT (DROP/DELETE/UPDATE/INSERT/;...)    #       gợi ý: kiểm tra sql.strip().lower().startswith('select') và không chứa ';'    raise NotImplementedErrordef search_policy(query: str, k: int = 3):    """Tra Sổ tay chính sách IT (RAG Ngày 9), trả về các đoạn + source_id."""    # TODO: gọi retrieve(query, k) và rút gọn thành text + source_id    raise NotImplementedErrordef calculator(expression: str):    """Tính biểu thức số học an toàn."""    # TODO: eval an toàn (chỉ cho +-*/(). số). Tránh eval trần.    raise NotImplementedErrordef today():    """Trả về ngày hôm nay dạng YYYY-MM-DD."""    # TODO    raise NotImplementedErrorTOOL_IMPL = {    "query_users_db": query_users_db,    "search_policy": search_policy,    "calculator": calculator,    "today": today,}

## 2) Khai báo tools schema cho OpenAI

In [ ]:
tools = [    {        "type": "function",        "function": {            "name": "query_users_db",            "description": "Chạy truy vấn SQL CHỈ-ĐỌC trên bảng users (danh bạ M365). "                             "Cột có dấu cách phải bọc nháy kép.",            "parameters": {                "type": "object",                "properties": {                    "sql": {"type": "string", "description": "Câu SELECT hợp lệ"}                },                "required": ["sql"],            },        },    },    # TODO: khai báo tương tự cho search_policy, calculator, today]

## 3) Vòng lặp agent

In [ ]:
def run_agent(question: str, max_steps: int = 5, verbose: bool = True):    """Trả về (câu trả lời cuối, trace các tool đã gọi)."""    messages = [        {"role": "system", "content": "Bạn là trợ lý. Khi cần dữ liệu người dùng "         "hãy gọi query_users_db; khi cần chính sách IT hãy gọi search_policy; "         "khi cần tính toán hãy gọi calculator. KHÔNG bịa số — hãy gọi tool."},        {"role": "user", "content": question},    ]    trace = []    for step in range(max_steps):        # TODO: gọi client.chat.completions.create(model=MODEL, messages=messages, tools=tools)        # TODO: nếu message.tool_calls: với mỗi call -> chạy TOOL_IMPL[name](**args),        #       append message.tool_calls vào messages, rồi append 1 message role='tool'        #       kèm ĐÚNG tool_call_id và content=json.dumps(result). Ghi vào trace.        # TODO: nếu KHÔNG có tool_calls -> đây là câu trả lời cuối, return (content, trace)        raise NotImplementedError    return "[Dừng vì vượt max_steps]", trace

## 4) Chạy thử trên ≥5 câu hỏi hỗn hợp

In [ ]:
questions = [    "Phòng ban nào có nhiều tài khoản nhất và bao nhiêu?",      # cần DB    "Bao lâu phải đổi mật khẩu theo chính sách IT?",           # cần policy    "Có bao nhiêu tài khoản bị chặn đăng nhập?",               # cần DB    "Tổng số tài khoản chia cho 2 bằng bao nhiêu?",            # cần DB + calculator (>=2 tool)    "Xin chào, bạn giúp được gì?",                             # KHÔNG cần tool]for q in questions:    print("=" * 80)    print("Q:", q)    answer_text, trace = run_agent(q)    print("Tool trace:", [t.get("name") for t in trace])    print("A:", answer_text)

## DoD — tự kiểm- [ ] Agent chạy ≥5 câu, in **trace tool call** cho mỗi câu.- [ ] ≥1 câu dùng **≥2 tool**; ≥1 câu **không cần tool** vẫn trả đúng.- [ ] `query_users_db` **chỉ cho SELECT** + **giới hạn `max_steps`**; không crash khi tool lỗi.- [ ] Restart & Run All không lỗi; key trong env var.